# Normalización del dataset limpiado por KNIME

**Archivo de entrada:** `Consulta_Externa_Muestra_Limpia.csv` (limpiado por KNIME)

**Archivo de salida:** `Consulta_Externa_Limpiada_KnimeAndPy.csv` (KNIME + normalización Python)

**Normalizaciones aplicadas:**
- Fechas: verificar/normalizar a formato `yyyy-mm-dd`
- Caracteres especiales: eliminar tildes/acentos restantes
- Encoding: corregir caracteres corruptos

## 1. Imports

In [1]:
import pandas as pd
import os
import gc
import unicodedata

## 2. Cargar el dataset limpio por KNIME

In [2]:
CARPETA = os.path.join(os.getcwd(), 'Muestras_Consulta_Externa', 'Consulta_Externa_Limpiada_PY')
ARCHIVO_ENTRADA = os.path.join(CARPETA, 'Consulta_Externa_Muestra_Limpia.csv')
ARCHIVO_SALIDA = os.path.join(CARPETA, 'Consulta_Externa_Limpiada_KnimeAndPy.csv')

print(f'Carpeta: {CARPETA}')
print(f'Archivo entrada: {ARCHIVO_ENTRADA}')
print(f'Existe: {os.path.exists(ARCHIVO_ENTRADA)}')

Carpeta: c:\Users\ASUS TUF\Desktop\DatasetsAnalisis\notebooks\Muestras_Consulta_Externa\Consulta_Externa_Limpiada_PY
Archivo entrada: c:\Users\ASUS TUF\Desktop\DatasetsAnalisis\notebooks\Muestras_Consulta_Externa\Consulta_Externa_Limpiada_PY\Consulta_Externa_Muestra_Limpia.csv
Existe: False


In [3]:
# Leer el CSV limpio por KNIME
# El archivo usa coma como separador, comillas como calificador de texto, y utf-8-sig (BOM)
df = pd.read_csv(
    ARCHIVO_ENTRADA,
    sep=',',
    encoding='utf-8-sig',
    dtype=str,
    quotechar='"',
    low_memory=False
)

print(f'Registros: {len(df):,}')
print(f'Columnas: {len(df.columns)}')
print(f'Columnas: {list(df.columns)}')

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\ASUS TUF\\Desktop\\DatasetsAnalisis\\notebooks\\Muestras_Consulta_Externa\\Consulta_Externa_Limpiada_PY\\Consulta_Externa_Muestra_Limpia.csv'

## 3. Funciones de normalización

Mismas funciones del Notebook 01, punto 4.

In [ ]:
def eliminar_tildes(texto):
    """Elimina tildes/acentos de un string usando Unicode NFD decomposition."""
    if not isinstance(texto, str):
        return texto
    nfd = unicodedata.normalize('NFD', texto)
    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')


def normalizar_fecha(fecha_str):
    """Convierte cualquier formato de fecha a yyyy-mm-dd.
    Soporta: dd/mm/yyyy, dd-mm-yyyy, yyyy-mm-dd, yyyy/mm/dd, etc.
    """
    if not isinstance(fecha_str, str) or pd.isna(fecha_str):
        return fecha_str
    fecha_str = fecha_str.strip()
    if fecha_str == '':
        return fecha_str

    separador = '/' if '/' in fecha_str else '-' if '-' in fecha_str else None
    if separador is None:
        return fecha_str

    partes = fecha_str.split(separador)
    if len(partes) != 3:
        return fecha_str

    p0, p1, p2 = partes

    if len(p0) == 4:
        anio, mes, dia = p0, p1.zfill(2), p2.zfill(2)
    elif len(p2) == 4:
        dia, mes, anio = p0.zfill(2), p1.zfill(2), p2
    else:
        return fecha_str

    return f'{anio}-{mes}-{dia}'


print('Funciones de normalización definidas.')

## 4. Verificar estado actual del dataset

Revisar qué normalizaciones aún son necesarias.

In [ ]:
# Columnas de fechas
COLS_FECHA = ['FECHA_ATENCION', 'FECHA_NACIMIENTO']

# Columnas de texto
COLS_TEXTO = [
    'ZONA', 'PROVINCIA', 'CANTON',
    'SEXO', 'NACIONALIDAD', 'AUTOIDENTIFICACION_ETNICA',
    'PROVINCIA_RESIDENCIA', 'CANTON_RESIDENCIA',
    'GRUPO_PRIORITARIO_1', 'GRUPO_PRIORITARIO_2', 'GRUPO_PRIORITARIO_3',
    'DESCRIPCION_CIE10_1', 'DESCRIPCION_CIE10_2', 'DESCRIPCION_CIE10_3',
    'TIPO_ATENCION1', 'TIPO_ATENCION2', 'TIPO_ATENCION3',
    'CONDICION_DIAGNOSTICO1', 'CONDICION_DIAGNOSTICO2', 'CONDICION_DIAGNOSTICO3'
]

# Verificar fechas: ¿todas están en yyyy-mm-dd?
print('=== Verificación de fechas ===')
for col in COLS_FECHA:
    sample = df[col].dropna()[:20]
    formato_ok = all(len(str(v).split('-')) == 3 for v in sample)
    print(f'  {col}: formato yyyy-mm-dd = {formato_ok} (ejemplo: {sample.iloc[0]})')

# Verificar caracteres especiales
print('\n=== Verificación de caracteres especiales ===')
for col in COLS_TEXTO:
    if col not in df.columns:
        continue
    vals = df[col].dropna().unique()
    problemas = []
    for v in vals:
        if any(ord(c) > 127 for c in str(v)):
            problemas.append(v)
    if problemas:
        print(f'  {col}: {len(problemas)} valores con caracteres no-ASCII')
        for p in problemas[:3]:
            print(f'    -> {repr(p)}')

print('\nVerificación completada.')

## 5. Aplicar normalizaciones

In [ ]:
# 1. Normalizar fechas a yyyy-mm-dd (aunque ya estén en ese formato, asegurar consistencia)
print('Normalizando fechas...')
for col in COLS_FECHA:
    antes = df[col].iloc[0]
    df[col] = df[col].apply(normalizar_fecha)
    despues = df[col].iloc[0]
    print(f'  {col}: "{antes}" -> "{despues}"')

# 2. Eliminar tildes/acentos en columnas de texto
print('\nEliminando tildes/acentos...')
for col in COLS_TEXTO:
    if col not in df.columns:
        continue
    df[col] = df[col].apply(eliminar_tildes)
print('  Tildes eliminadas en todas las columnas de texto.')

# 3. Strip espacios extra (margen de seguridad)
print('\nLimpiando espacios extra...')
for col in COLS_TEXTO:
    if col not in df.columns:
        continue
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
print('  Espacios limpiados.')

# 4. Corregir datos sucios residuales que KNIME no detecto
print('\nCorrigiendo datos sucios residuales...')
col_auto = 'AUTOIDENTIFICACION_ETNICA'
if col_auto in df.columns:
    antes = df[col_auto].value_counts()

    # Corregir: "NEGRO /A" (espacio antes de /A) -> "NEGRO/A"
    df[col_auto] = df[col_auto].replace('NEGRO /A', 'NEGRO/A')

    # Corregir: "AFROECUATORIANO/AFRODESCENDIENTEFRODESCENDIENTE" (duplicado)
    df[col_auto] = df[col_auto].replace(
        'AFROECUATORIANO/AFRODESCENDIENTEFRODESCENDIENTE',
        'AFROECUATORIANO/AFRODESCENDIENTE'
    )

    despues = df[col_auto].value_counts()

    if 'NEGRO /A' in antes.index:
        print(f'  NEGRO /A  ->  NEGRO/A ({int(antes["NEGRO /A"]):,} registros)')
    if 'AFROECUATORIANO/AFRODESCENDIENTEFRODESCENDIENTE' in antes.index:
        n = int(antes['AFROECUATORIANO/AFRODESCENDIENTEFRODESCENDIENTE'])
        print(f'  AFRO...EFRODESCENDIENTE -> AFROECUATORIANO/AFRODESCENDIENTE ({n:,} registros)')
    print(f'  Valores unicos AUTOID antes: {len(antes)}, despues: {len(despues)}')

print('\nNormalizaciones completadas.')

## 6. Verificar resultado

In [ ]:
# Verificar que no quedan caracteres especiales
print('=== Verificación post-normalización ===')
problemas_restantes = 0
for col in COLS_TEXTO:
    if col not in df.columns:
        continue
    vals = df[col].dropna().unique()
    for v in vals:
        if any(ord(c) > 127 for c in str(v)):
            print(f'  Pendiente: {col} -> {repr(v)}')
            problemas_restantes += 1

if problemas_restantes == 0:
    print('  Sin caracteres especiales. Todo OK.')

# Verificar datos sucios residuales corregidos
print('\n=== Verificacion de datos sucios residuales ===')
col_auto = 'AUTOIDENTIFICACION_ETNICA'
if col_auto in df.columns:
    vals = set(df[col_auto].dropna().unique())
    if 'NEGRO /A' in vals:
        print('  PENDIENTE: NEGRO /A todavia existe')
    else:
        print('  OK: NEGRO /A corregido a NEGRO/A')
    if 'AFROECUATORIANO/AFRODESCENDIENTEFRODESCENDIENTE' in vals:
        print('  PENDIENTE: AFRO...EFRODESCENDIENTE todavia existe')
    else:
        print('  OK: AFRO...EFRODESCENDIENTE corregido a AFROECUATORIANO/AFRODESCENDIENTE')

print(f'\nRegistros: {len(df):,}')
print(f'Columnas: {len(df.columns)}')
print(f'\nMuestra de AUTOID: {list(df["AUTOIDENTIFICACION_ETNICA"].unique())}')
print(f'Muestra de PROVINCIAS: {list(df["PROVINCIA"].unique())[:10]}')
print(f'Muestra FECHA_ATENCION: {list(df["FECHA_ATENCION"][:3])}')

## 7. Guardar el dataset final

In [ ]:
# Guardar como Consulta_Externa_Limpiada_KnimeAndPy.csv
df.to_csv(ARCHIVO_SALIDA, index=False, encoding='utf-8')
print(f'Guardado: {ARCHIVO_SALIDA}')
print(f'Registros: {len(df):,}')
print(f'Columnas: {len(df.columns)}')

# Liberar memoria
del df
gc.collect()

print('\nProceso completado.')

## 8. Descargar el archivo

In [ ]:
# Verificar que el archivo existe y mostrar información
if os.path.exists(ARCHIVO_SALIDA):
    size_mb = os.path.getsize(ARCHIVO_SALIDA) / (1024 * 1024)
    print(f'Archivo listo para descargar:')
    print(f'  Nombre: Consulta_Externa_Limpiada_KnimeAndPy.csv')
    print(f'  Ubicación: {ARCHIVO_SALIDA}')
    print(f'  Tamaño: {size_mb:.1f} MB')
    print()
    # Abrir la carpeta en el explorador de archivos
    os.startfile(CARPETA)
    print('Carpeta abierta en el explorador de archivos.')
else:
    print('ERROR: El archivo no fue encontrado.')